In [1]:
# Task 4.1
from pymongo import MongoClient
import json


# Loading data from 'prelim_results.json' to a NoSQL database

def refresh_and_load_db():
    client = MongoClient()
    db = client["DHS"]
    coll = db["Prelims"]
    
    with open("prelim_results.json") as f:
        data = json.load(f)

    coll.delete_many({})
    coll.insert_many(data)

refresh_and_load_db()

In [ ]:
# Task 4.1 Cont'd [Flask]

from pymongo import MongoClient
from flask import Flask,render_template,request

def parse(student):
    parsed = [
        student["student_id"],
        student["full_name"]
    ] + sum([
        list(i) for i in student["results"].items()
    ],[])

    return parsed

def get_results(student=None):
    client = MongoClient()
    db = client["DHS"]
    coll = db["Prelims"]

    query_doc = {"full_name": student} if student is not None else {}
    data = coll.find(query_doc,{"_id":0})

    parsed_data = [parse(i) for i in list(data)]
    return parsed_data

def get_namelist():
    client = MongoClient()
    db = client["DHS"]
    coll = db["Prelims"]
    data = coll.find({},{"full_name":1,"_id":0})
    return [i["full_name"] for i in list(data)]

app = Flask(__name__)

@app.route('/',methods=["GET","POST"])
def index():
    namelist = get_namelist()
    name = request.form.get("student")
    # if name not in namelist:
    #     name = None
    return render_template("index.html",namelist=namelist, results = get_results(name))


app.run()

UsageError: Line magic function `%%writefile` not found.


In [3]:
%run app.py


 * Serving Flask app 'app'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
